In [1]:
import os, glob, json, hashlib, subprocess, datetime
import numpy as np, pandas as pd
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
def run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError((r.stdout + r.stderr).replace(token, "***"))
auth = f"https://{token}@github.com/{USER}/{REPO}.git"
os.chdir("/kaggle/working")
run(["rm", "-rf", REPO])
run(["git", "clone", "-q", "-b", "main", auth, REPO])
run(["git", "remote", "set-url", "origin", f"https://github.com/{USER}/{REPO}.git"], cwd=REPO)

npz = glob.glob("/kaggle/input/**/brain_mri_curated_v1.npz", recursive=True)
assert npz, "npz not found, check whether Kaggle unpacked it on upload"
d = np.load(npz[0])
ids, imgs, labels, masks = d["image_ids"], d["images"], d["labels"], d["masks"]
h = pd.read_csv(f"{REPO}/data_manifest/hashes.csv").set_index("image_id")
sp = pd.read_csv(f"{REPO}/data_manifest/splits_v1.csv")

mismatch = sum(hashlib.sha256(imgs[k].tobytes()).hexdigest() != h.at[i, "array_sha256"]
               for k, i in enumerate(ids))
fs = sp.source.values == "figshare"
checks = {
    "verified_on": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d"),
    "n_images": int(len(ids)),
    "rows_match_splits_v1": bool(len(ids) == len(sp)),
    "order_matches_splits_v1": bool((ids == sp.image_id.values).all()),
    "labels_match_splits_v1": bool((labels == sp.label_id.values).all()),
    "array_hash_mismatches": int(mismatch),
    "figshare_masks_nonempty": int((masks[fs].reshape(fs.sum(), -1).sum(1) > 0).sum()),
    "shape": list(imgs.shape), "dtype": str(imgs.dtype),
}
print(json.dumps(checks, indent=2))

json.dump(checks, open(f"{REPO}/data_manifest/curated_checks.json", "w"), indent=2)
run(["git", "config", "user.name", USER], cwd=REPO)
run(["git", "config", "user.email", EMAIL], cwd=REPO)
run(["git", "add", "-A"], cwd=REPO)
run(["git", "commit", "-q", "-m", "Step 1.4 curated dataset hash verification"], cwd=REPO)
run(["git", "push", "-q", auth, "main"], cwd=REPO)
print("Verification pushed to main.")

{
  "verified_on": "2026-10-06",
  "n_images": 3845,
  "rows_match_splits_v1": true,
  "order_matches_splits_v1": true,
  "labels_match_splits_v1": true,
  "array_hash_mismatches": 0,
  "figshare_masks_nonempty": 3010,
  "shape": [
    3845,
    128,
    128
  ],
  "dtype": "uint8"
}
Verification pushed to main.
